# Detección de Fraude en Tarjetas de Crédito con Redes Neuronales

---

**Objetivo:** Construir un pipeline completo de preparación de datos con **Polars** y entrenar una red neuronal con **Keras** para detectar transacciones fraudulentas.

**Dataset:** IBM Synthetic Credit Card Transactions — transacciones sintéticas realistas con etiquetas de fraude verificadas.

**Lo que aprenderás en este notebook:**
1. Cómo leer y transformar datos tabulares con **Polars** (un DataFrame moderno y muy rápido)
2. Por qué el **orden** del pipeline importa para evitar *data leakage* (filtración de información del futuro hacia el modelo)
3. Cómo construir **features** significativas a partir de datos crudos: fechas, categorías, montos
4. El **problema del desbalance de clases** y cómo atacarlo
5. Cómo diseñar, entrenar y evaluar una red neuronal en **Keras** para clasificación binaria
6. Por qué el *accuracy* es una métrica engañosa en detección de fraude

---

> *"Un modelo que siempre predice 'no fraude' puede tener 99% de accuracy — y ser completamente inútil."*

## Configuración inicial

El dataset completo (`credit_card_transactions-ibm_v2.csv`) es pesado. Para pruebas rápidas durante el desarrollo, puedes usar el subconjunto de un solo usuario (`User0_credit_card_transactions.csv`). Cambia la variable `MODO_PRUEBA` según necesites.

| Archivo | Filas aprox. | Uso recomendado |
|---|---|---|
| `User0_credit_card_transactions.csv` | ~20 000 | Desarrollo y pruebas rápidas |
| `credit_card_transactions-ibm_v2.csv` | ~24 000 000 | Entrenamiento real |

In [ ]:
import polars as pl
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report
)

import keras
from keras import layers

np.random.seed(42)

# ─── Selección del dataset ────────────────────────────────────────────────────
MODO_PRUEBA = False

ARCHIVO = (
    "data/User0_credit_card_transactions.csv"
    if MODO_PRUEBA
    else "data/credit_card_transactions-ibm_v2.csv"
)
print(f"Usando: {ARCHIVO}")

---
## 1. El problema: detección de fraude

La detección de fraude es un problema de **clasificación binaria supervisada**: dada una transacción, predecir si es fraudulenta (`Is Fraud? = Yes`) o legítima (`Is Fraud? = No`).

### ¿Por qué es difícil?

1. **Desbalance extremo de clases**: el fraude representa típicamente entre 0.1% y 1% de las transacciones. Un modelo que siempre responde "no fraude" tendría 99%+ de *accuracy* — pero detectaría cero fraudes.

2. **Los fraudes evolucionan**: los patrones de fraude cambian con el tiempo; lo que detecta bien el modelo hoy puede no funcionar mañana.

3. **El costo asimétrico del error**: un falso negativo (fraude no detectado) es mucho más costoso que un falso positivo (transacción legítima bloqueada).

### Las métricas correctas

En lugar de *accuracy*, usaremos:

| Métrica | Fórmula | Qué mide |
|---|---|---|
| **Recall** | $\frac{TP}{TP + FN}$ | ¿Qué fracción del fraude real detectamos? |
| **Precision** | $\frac{TP}{TP + FP}$ | ¿Qué fracción de nuestras alertas son reales? |
| **AUC-ROC** | Área bajo curva ROC | Capacidad discriminativa global |
| **AUC-PR** | Área bajo curva PR | Más informativa que ROC en datasets desbalanceados |

donde $TP$ = verdaderos positivos, $FN$ = falsos negativos, $FP$ = falsos positivos.

---
## 2. Carga y exploración del dataset con Polars

### ¿Por qué Polars en lugar de Pandas?

| Característica | Pandas | Polars |
|---|---|---|
| Motor de ejecución | Python / NumPy | Rust (SIMD, multihilo) |
| Velocidad en datasets grandes | Lento | 5–20× más rápido |
| Memoria | Copia frecuente | Copy-on-write eficiente |
| API lazy (diferida) | No | `scan_csv` → ejecuta solo al hacer `.collect()` |
| Manejo de nulos | `NaN` y `None` mezclados | Un solo tipo: `null` |

Para datasets de millones de filas, usamos **LazyFrame** con `scan_csv`: Polars construye un plan de ejecución optimizado y solo lee/procesa los datos cuando se llama a `.collect()`.

In [ ]:
# scan_csv crea un LazyFrame — no carga nada en memoria todavía
lf_raw = pl.scan_csv(ARCHIVO, null_values=["", "NA", "nan"])

# .collect_schema() inspecciona los tipos sin leer filas
schema = lf_raw.collect_schema()
print("Columnas y tipos inferidos por Polars:")
print("─" * 45)
for nombre, tipo in schema.items():
    print(f"  {nombre:<25} {str(tipo):<15}")

In [ ]:
# Ahora sí ejecutamos la lectura completa
df_raw = lf_raw.collect()

n_filas, n_cols = df_raw.shape
print(f"Dimensiones: {n_filas:>12,} filas × {n_cols} columnas")
print()
print("Primeras 3 filas:")
print(df_raw.head(3))
print()
print("Valores nulos por columna:")
nulos = df_raw.null_count()
print(nulos)

### 2.1 Distribución de la variable objetivo

El primer paso siempre es entender el desbalance. Veamos cuántas transacciones son fraude.

In [ ]:
dist_fraude = (
    df_raw
    .group_by("Is Fraud?")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).alias("porcentaje")
    )
    .sort("Is Fraud?")
)
print(dist_fraude)

counts = dist_fraude["count"].to_list()
labels = dist_fraude["Is Fraud?"].to_list()
pcts   = dist_fraude["porcentaje"].to_list()

fig = go.Figure(go.Bar(
    x=labels,
    y=counts,
    text=[f"{c:,}<br>({p:.2f}%)" for c, p in zip(counts, pcts)],
    textposition="outside",
    marker_color=["#2ecc71", "#e74c3c"],
    marker_line_width=0,
))
fig.update_layout(
    title=dict(text="Distribución de la variable objetivo: fraude vs. legítima", font_size=16),
    xaxis_title="¿Es fraude?",
    yaxis_title="Número de transacciones",
    plot_bgcolor="#f8f9fa",
    width=550, height=420,
    yaxis=dict(type="log"),
)
fig.add_annotation(
    text="Eje Y en escala logarítmica<br>para apreciar la diferencia",
    x=1, y=0.1, xref="paper", yref="paper",
    showarrow=False, font=dict(color="gray", size=11)
)
fig.show()

### 2.2 Distribución del monto por clase

Antes de transformar, veamos si el monto tiene una distribución diferente entre fraude y transacciones legítimas — esto nos indica si es una feature útil.

In [ ]:
# Parsear Amount para poder graficar (quitar el símbolo $)
df_eda = df_raw.with_columns(
    pl.col("Amount").str.strip_chars("$").cast(pl.Float32).alias("amount_num")
)

legit  = df_eda.filter(pl.col("Is Fraud?") == "No")["amount_num"].to_numpy()
fraude = df_eda.filter(pl.col("Is Fraud?") == "Yes")["amount_num"].to_numpy()

# Muestrear legítimas para que la visualización sea manejable
rng = np.random.default_rng(42)
legit_sample = rng.choice(legit, size=min(50_000, len(legit)), replace=False)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=legit_sample, name="Legítima",
    histnorm="probability density",
    marker_color="#2ecc71", opacity=0.65,
    nbinsx=100, xbins=dict(start=0, end=500)
))
fig.add_trace(go.Histogram(
    x=fraude, name="Fraude",
    histnorm="probability density",
    marker_color="#e74c3c", opacity=0.65,
    nbinsx=100, xbins=dict(start=0, end=500)
))
fig.update_layout(
    barmode="overlay",
    title="Distribución del monto: fraude vs. legítima (0–500 $)",
    xaxis_title="Monto ($)",
    yaxis_title="Densidad de probabilidad",
    plot_bgcolor="#f8f9fa",
    width=700, height=400,
    legend=dict(x=0.75, y=0.95)
)
fig.show()

print(f"Monto medio — Legítima: ${np.mean(legit):.2f} | Fraude: ${np.mean(fraude):.2f}")
print(f"Monto mediana — Legítima: ${np.median(legit):.2f} | Fraude: ${np.median(fraude):.2f}")

### 2.3 Patrones temporales del fraude

¿El fraude ocurre más en ciertos horarios?

In [ ]:
df_hora = (
    df_raw
    .with_columns(
        pl.col("Time").str.split(":").list.get(0).cast(pl.Int8).alias("hora"),
        (pl.col("Is Fraud?") == "Yes").cast(pl.Int8).alias("es_fraude")
    )
    .group_by("hora")
    .agg(
        pl.col("es_fraude").sum().alias("fraudes"),
        pl.len().alias("total")
    )
    .with_columns(
        (pl.col("fraudes") / pl.col("total") * 100).alias("tasa_fraude_pct")
    )
    .sort("hora")
)

horas = df_hora["hora"].to_list()
tasas = df_hora["tasa_fraude_pct"].to_list()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=horas, y=tasas, mode="lines+markers",
    line=dict(color="#e74c3c", width=2.5),
    marker=dict(size=8, color="#c0392b"),
    fill="tozeroy", fillcolor="rgba(231,76,60,0.12)"
))
fig.update_layout(
    title="Tasa de fraude por hora del día",
    xaxis=dict(title="Hora del día", tickmode="linear", dtick=2),
    yaxis=dict(title="Tasa de fraude (%)"),
    plot_bgcolor="#f8f9fa",
    width=700, height=380
)
fig.show()

---
## 3. Ingeniería de features con Polars

La **ingeniería de features** es el proceso de transformar los datos crudos en representaciones que la red neuronal pueda aprender eficazmente. Una red neuronal solo entiende números en rangos similares — no entiende `"$134.09"`, `"06:21"` ni `"Swipe Transaction"` en su forma original.

### El pipeline completo

```
Datos crudos
    │
    ├─ 1. Parseo de Amount ($  → float)
    ├─ 2. Parseo de Time (HH:MM → hora, minuto)
    ├─ 3. Codificación cíclica (hora, mes, día → sin/cos)
    ├─ 4. One-hot de Use Chip (Swipe / Chip / Online)
    ├─ 5. MCC por rango de frecuencia
    └─ 6. Flag binario de Errors?
        │
        ↓ SPLIT TRAIN / TEST ← ¡AQUÍ, antes del target encoding!
        │
        └─ 7. Target encoding de Merchant State (solo fit en train)
```

> **Nota crítica:** el paso 7 (target encoding) se aplica **después** de la división train/test. Calcularlo sobre todo el dataset produciría *data leakage*: el modelo "vería" información del conjunto de validación durante el entrenamiento, dando resultados artificialmente optimistas.

### 3.1 Parseo de Amount y target

El monto viene como string con prefijo `$` (ej. `"$134.09"`). Lo convertimos a `Float32`. También convertimos la columna objetivo `Is Fraud?` de `Yes/No` a `1/0`.

Usamos `Float32` (en lugar de `Float64`) para reducir el consumo de memoria a la mitad — suficiente precisión para montos de tarjeta.

In [ ]:
df = (
    df_raw
    .with_columns([
        # Amount: quitar '$' y convertir a número
        pl.col("Amount").str.strip_chars("$").cast(pl.Float32).alias("amount"),

        # Target: Yes → 1, No → 0
        (pl.col("Is Fraud?") == "Yes").cast(pl.Int8).alias("target"),
    ])
)

print("Estadísticas del monto:")
print(df.select("amount").describe())
print()
print(f"Fraudes en el dataset: {df['target'].sum():,} de {len(df):,} ({df['target'].mean()*100:.3f}%)")

### 3.2 Features temporales

La columna `Time` contiene la hora en formato `"HH:MM"`. Extraemos:
- `hour`: la hora del día (0–23)
- `minute`: el minuto (0–59)

También guardamos `Year` como feature numérico directamente — puede capturar deriva temporal (el fraude cambia con los años).

In [ ]:
df = (
    df
    .with_columns([
        # Dividir "HH:MM" por ":" y tomar cada parte
        pl.col("Time").str.split(":").list.get(0).cast(pl.Int8).alias("hour"),
        pl.col("Time").str.split(":").list.get(1).cast(pl.Int8).alias("minute"),
    ])
)

print(df.select(["Time", "hour", "minute"]).head(5))

### 3.3 Codificación cíclica para variables periódicas

Las variables temporales son **cíclicas**: la hora 23 y la hora 0 son contiguas, el mes 12 y el mes 1 también. Si las dejamos como números enteros, la red pensará que 23 y 0 están en extremos opuestos del espacio de features.

La solución: proyectar cada variable en un **círculo** usando seno y coseno.

Para una variable $x$ con período $T$:

$$x_{\sin} = \sin\!\left(\frac{2\pi\, x}{T}\right) \qquad x_{\cos} = \cos\!\left(\frac{2\pi\, x}{T}\right)$$

Así, hora 23 y hora 0 quedan próximas en el espacio $(x_{\sin}, x_{\cos})$:

| Variable | Período $T$ |
|---|---|
| Hora del día | 24 |
| Mes del año | 12 |
| Día del mes | 31 |

In [ ]:
TWO_PI = 2 * np.pi

df = (
    df
    .with_columns([
        # Hora cíclica (período 24)
        (pl.col("hour") * (TWO_PI / 24)).sin().alias("hour_sin"),
        (pl.col("hour") * (TWO_PI / 24)).cos().alias("hour_cos"),

        # Mes cíclico (período 12)
        (pl.col("Month") * (TWO_PI / 12)).sin().alias("month_sin"),
        (pl.col("Month") * (TWO_PI / 12)).cos().alias("month_cos"),

        # Día del mes cíclico (período 31)
        (pl.col("Day") * (TWO_PI / 31)).sin().alias("day_sin"),
        (pl.col("Day") * (TWO_PI / 31)).cos().alias("day_cos"),
    ])
)

# ─── Visualización: ¿qué hace la codificación cíclica? ────────────────────────
horas = np.arange(24)
h_sin = np.sin(horas * TWO_PI / 24)
h_cos = np.cos(horas * TWO_PI / 24)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Hora como número entero (0–23)", "Hora como punto en el círculo (sin, cos)"),
    column_widths=[0.45, 0.55]
)

# Representación lineal — el problema
fig.add_trace(go.Scatter(
    x=horas, y=[0] * 24, mode="markers+text",
    marker=dict(size=10, color=horas, colorscale="Plasma"),
    text=[str(h) for h in horas], textposition="top center",
    textfont=dict(size=9), showlegend=False
), row=1, col=1)
fig.add_annotation(
    x=11.5, y=-0.3, text="hora 0 y hora 23<br>parecen muy lejanas ❌",
    showarrow=False, font=dict(color="#e74c3c", size=11), row=1, col=1
)

# Representación circular — la solución
fig.add_trace(go.Scatter(
    x=h_cos, y=h_sin, mode="markers+text",
    marker=dict(size=12, color=horas, colorscale="Plasma",
                showscale=True, colorbar=dict(title="hora", thickness=12, x=1.05)),
    text=[str(h) for h in horas], textposition="top center",
    textfont=dict(size=9), showlegend=False
), row=1, col=2)
# Círculo de referencia
theta = np.linspace(0, TWO_PI, 200)
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta), mode="lines",
    line=dict(color="lightgray", dash="dot"), showlegend=False
), row=1, col=2)
fig.add_annotation(
    x=0, y=-1.3, text="hora 0 y hora 23<br>quedan adyacentes ✅",
    showarrow=False, font=dict(color="#2ecc71", size=11), row=1, col=2
)

fig.update_layout(
    height=380, width=860,
    title_text="Por qué usar codificación cíclica para variables periódicas",
    title_font_size=15,
    plot_bgcolor="#f8f9fa"
)
fig.update_xaxes(title_text="Hora (entero)", row=1, col=1)
fig.update_xaxes(title_text="cos(hora)", row=1, col=2)
fig.update_yaxes(title_text="sin(hora)", scaleanchor="x2", scaleratio=1, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=1)
fig.show()

### 3.4 Codificación de `Use Chip` (one-hot encoding)

La columna `Use Chip` indica el método de transacción con tres valores posibles:
- `"Swipe Transaction"` — deslizamiento de la banda magnética
- `"Chip Transaction"` — chip EMV
- `"Online Transaction"` — compra por internet

Las redes neuronales no pueden operar sobre strings. El **one-hot encoding** crea una columna binaria por categoría. Con 3 categorías solo necesitamos **2 columnas** (la tercera se infiere implícitamente — se llama categoría de referencia):

| Use Chip | chip_swipe | chip_chip |
|---|---|---|
| Swipe Transaction | 1 | 0 |
| Chip Transaction | 0 | 1 |
| Online Transaction | 0 | 0 | ← categoría de referencia

> **¿Por qué no 3 columnas?** Si creamos una columna por cada categoría, la suma de las tres siempre es 1 — son linealmente dependientes. Esto se llama *dummy variable trap* y puede crear problemas numéricos.

In [ ]:
print("Valores únicos de 'Use Chip':")
print(df["Use Chip"].value_counts().sort("count", descending=True))
print()

df = (
    df
    .with_columns([
        (pl.col("Use Chip") == "Swipe Transaction").cast(pl.Int8).alias("chip_swipe"),
        (pl.col("Use Chip") == "Chip Transaction").cast(pl.Int8).alias("chip_chip"),
        # "Online Transaction" = chip_swipe=0 y chip_chip=0
    ])
)

print("Verificación del encoding:")
print(
    df.select(["Use Chip", "chip_swipe", "chip_chip"])
    .unique()
    .sort("Use Chip")
)

### 3.5 MCC: codificación por rango de frecuencia

El **Merchant Category Code (MCC)** es un código numérico de 4 dígitos que clasifica el tipo de comercio (supermercado, farmacia, gasolinera, etc.). En este dataset puede haber decenas de categorías únicas.

**Problema:** no podemos usar el MCC directamente como número porque `5300 > 5411` no significa nada (son solo etiquetas). Y crear una columna por cada MCC (one-hot) generaría demasiadas dimensiones.

**Solución — codificación por frecuencia:** asignamos a cada MCC un rango según cuántas veces aparece en el dataset. El MCC más frecuente recibe rango 0, el siguiente rango 1, etc. Esto:
- Preserva información ordinal útil (comercios comunes vs. raros)
- No explota la dimensionalidad
- Es robusto a nuevos valores en producción

In [ ]:
print(f"Categorías MCC únicas: {df['MCC'].n_unique()}")
print()
print("MCCs más frecuentes:")
print(df["MCC"].value_counts().sort("count", descending=True).head(8))

# Construir tabla de rangos: MCC más frecuente → rango 0
mcc_freq = (
    df
    .group_by("MCC")
    .agg(pl.len().alias("mcc_count"))
    .sort("mcc_count", descending=True)
    .with_row_index("mcc_rank")  # índice de fila = rango
    .select(["MCC", "mcc_rank"])
)

df = df.join(mcc_freq, on="MCC", how="left")

print()
print("Rango asignado a cada MCC:")
print(df.select(["MCC", "mcc_rank"]).unique().sort("mcc_rank").head(6))

### 3.6 Flag de errores

La columna `Errors?` registra si hubo algún error en la transacción (ej. PIN incorrecto, tarjeta rechazada). La mayoría de las filas son `null` (sin error).

En lugar de intentar codificar el tipo de error (alta cardinalidad), simplemente creamos un **flag binario**: `1` si hubo algún error, `0` si no. Esta información puede ser relevante — transacciones con errores previos tienen un patrón diferente.

In [ ]:
print("Tipos de errores registrados:")
print(df["Errors?"].value_counts().sort("count", descending=True))

df = (
    df
    .with_columns(
        pl.col("Errors?").is_not_null().cast(pl.Int8).alias("has_error")
    )
)

print()
print("Distribución del flag has_error:")
print(df["has_error"].value_counts())

---
## 4. División train / test y desbalance de clases

### ¿Por qué dividir ahora y no al final?

Hay un paso pendiente (el target encoding del estado del comercio) que **requiere saber qué filas son de entrenamiento**. Si lo calculamos sobre todo el dataset y luego dividimos, estamos usando información del conjunto de test para construir las features — esto se llama **data leakage** y produce modelos artificialmente optimistas que fallan en producción.

### Estratificación

Como el fraude es muy raro, usamos `stratify=y` en el split: garantiza que la proporción de fraude sea la misma en train y en test. Sin esto, podría ocurrir que el test no tuviera casi fraudes — y la evaluación sería inútil.

In [ ]:
# Añadir índice de fila para poder dividir manteniendo el LazyFrame
df = df.with_row_index("row_id")

# Extraer índices y target para hacer el split
ids    = df["row_id"].to_numpy()
target = df["target"].to_numpy()

idx_train, idx_test = train_test_split(
    ids,
    test_size=0.20,
    random_state=42,
    stratify=target          # ← mantiene la proporción de fraude
)

df_train = df.filter(pl.col("row_id").is_in(idx_train))
df_test  = df.filter(pl.col("row_id").is_in(idx_test))

print(f"Train: {len(df_train):>10,} filas — fraude: {df_train['target'].mean()*100:.3f}%")
print(f"Test:  {len(df_test):>10,} filas — fraude: {df_test['target'].mean()*100:.3f}%")
print()
print("La proporción de fraude es casi idéntica en ambos conjuntos ✅")

### 4.1 El desbalance de clases — cálculo del peso de clase

La estrategia más simple para manejar el desbalance es darle más **peso** a los ejemplos de la clase minoritaria durante el entrenamiento. Si el fraude representa el 0.5% de los datos, el modelo debería penalizar 200 veces más un falso negativo que si las clases estuvieran balanceadas.

La fórmula es:

$$w_{clase} = \frac{N_{total}}{N_{clases} \times N_{clase}}$$

donde $N_{clase}$ es el número de ejemplos en esa clase. Keras acepta estos pesos en el parámetro `class_weight` de `.fit()`.

In [ ]:
y_train_raw = df_train["target"].to_numpy()
n_total = len(y_train_raw)
n_neg   = int((y_train_raw == 0).sum())   # legítimas
n_pos   = int((y_train_raw == 1).sum())   # fraudes

w_neg = n_total / (2 * n_neg)
w_pos = n_total / (2 * n_pos)
class_weight = {0: w_neg, 1: w_pos}

print("Conteos en train:")
print(f"  Legítimas: {n_neg:>10,}")
print(f"  Fraudes:   {n_pos:>10,}")
print(f"  Ratio:     {n_neg/n_pos:.0f}:1")
print()
print("Pesos de clase calculados:")
print(f"  Clase 0 (legítima):  {w_neg:.4f}")
print(f"  Clase 1 (fraude):    {w_pos:.4f}")
print()

# Visualizar el desbalance
fig = go.Figure(go.Pie(
    labels=["Legítima", "Fraude"],
    values=[n_neg, n_pos],
    hole=0.5,
    marker_colors=["#2ecc71", "#e74c3c"],
    textinfo="label+percent",
    textfont_size=14,
))
fig.update_layout(
    title=dict(text="Desbalance de clases en el conjunto de entrenamiento", font_size=15),
    annotations=[dict(text=f"{n_neg/n_pos:.0f}:1", x=0.5, y=0.5, font_size=22, showarrow=False)],
    width=480, height=380
)
fig.show()

### 4.2 Target encoding del estado del comercio (fit solo en train)

El **target encoding** reemplaza cada categoría por la **tasa de fraude media** de ese grupo. Es muy informativo porque captura directamente la señal de fraude por estado.

$$\text{state\_fraud\_rate}[s] = \frac{\sum_{i \in \text{train},\, \text{state}_i = s} y_i}{|\{i \in \text{train} : \text{state}_i = s\}|}$$

**Importantísimo:** calculamos esta media **solo sobre `df_train`** y luego la aplicamos a `df_test`. Si la calculáramos sobre todo el dataset, los datos de test "contaminarían" el encoding — data leakage.

In [ ]:
# Calcular tasa de fraude por estado — SOLO sobre train
state_target_enc = (
    df_train
    .group_by("Merchant State")
    .agg(pl.col("target").mean().alias("state_fraud_rate"))
)

# La tasa global de train se usa para estados no vistos en train (imputación)
global_fraud_rate = float(df_train["target"].mean())
print(f"Tasa global de fraude en train: {global_fraud_rate*100:.4f}%")
print()
print("Target encoding por estado (top 10 con más fraude):")
print(
    state_target_enc
    .sort("state_fraud_rate", descending=True)
    .head(10)
)

# Aplicar el encoding a train y test
df_train = df_train.join(state_target_enc, on="Merchant State", how="left")
df_test  = df_test.join(state_target_enc,  on="Merchant State", how="left")

# Rellenar nulos (estados no vistos en train) con la tasa global
df_train = df_train.with_columns(
    pl.col("state_fraud_rate").fill_null(global_fraud_rate)
)
df_test = df_test.with_columns(
    pl.col("state_fraud_rate").fill_null(global_fraud_rate)
)

print()
print("Valores nulos en state_fraud_rate después de imputación:")
print(f"  Train: {df_train['state_fraud_rate'].null_count()} | Test: {df_test['state_fraud_rate'].null_count()}")

---
## 5. Selección de features y normalización

### Features seleccionadas

| Feature | Descripción | Tipo |
|---|---|---|
| `amount` | Monto de la transacción (USD) | Numérica continua |
| `has_error` | ¿Hubo errores en la transacción? | Binaria |
| `hour_sin`, `hour_cos` | Hora del día (cíclica) | Numérica |
| `month_sin`, `month_cos` | Mes del año (cíclico) | Numérica |
| `day_sin`, `day_cos` | Día del mes (cíclico) | Numérica |
| `chip_swipe`, `chip_chip` | Método de transacción (one-hot) | Binaria |
| `mcc_rank` | Rango de frecuencia del tipo de comercio | Ordinal |
| `state_fraud_rate` | Tasa histórica de fraude del estado | Numérica |
| `Card` | ID de la tarjeta del usuario | Numérica discreta |
| `Year` | Año de la transacción (deriva temporal) | Numérica |

**Columnas descartadas:**
- `Merchant Name`: ID hasheado, altísima cardinalidad, necesitaría embeddings
- `Merchant City`: alta cardinalidad, señal parcialmente capturada por el estado
- `Zip`: incompleto (muchos nulos) y alta cardinalidad
- `User`: identificador del usuario, no feature de transacción

### Normalización con StandardScaler

Las redes neuronales son sensibles a la escala de los inputs. Una feature que vale `50,000` (monto grande) dominaría a otra que vale `0.8` (hora normalizada). El **StandardScaler** transforma cada feature para que tenga media 0 y desviación estándar 1:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

**Regla de oro:** el scaler se ajusta (`fit`) **solo sobre train** y se aplica (`transform`) a ambos conjuntos. Si ajustamos sobre todo el dataset, la media y la desviación de test contaminan los parámetros — de nuevo, data leakage.

In [ ]:
FEATURE_COLS = [
    "amount",
    "has_error",
    "hour_sin",  "hour_cos",
    "month_sin", "month_cos",
    "day_sin",   "day_cos",
    "chip_swipe", "chip_chip",
    "mcc_rank",
    "state_fraud_rate",
    "Card",
    "Year",
]

X_train_raw = df_train.select(FEATURE_COLS).to_numpy()
X_test_raw  = df_test.select(FEATURE_COLS).to_numpy()
y_train     = df_train["target"].to_numpy().astype("float32")
y_test      = df_test["target"].to_numpy().astype("float32")

# Fit solo sobre train → transform a ambos
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_test  = scaler.transform(X_test_raw).astype("float32")

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print()

# Verificar que las features de train tienen media ≈ 0 y std ≈ 1
print("Media de las features en train (debe ser ≈ 0):")
print(np.round(X_train.mean(axis=0), 4))
print()
print("Std de las features en train (debe ser ≈ 1):")
print(np.round(X_train.std(axis=0), 4))

---
## 6. Arquitectura de la red neuronal

Diseñamos una red **feedforward** (completamente conectada) con la siguiente estructura:

```
Entrada (14 features)
    │
    ├── Dense(64) + ReLU
    ├── BatchNormalization
    ├── Dropout(0.3)         ← apaga 30% de neuronas aleatoriamente en cada batch
    │
    ├── Dense(32) + ReLU
    ├── BatchNormalization
    ├── Dropout(0.2)
    │
    └── Dense(1) + Sigmoid   ← salida en [0,1]: probabilidad de fraude
```

### Decisiones de diseño

**Función de activación ReLU** — $\text{ReLU}(z) = \max(0, z)$. Es simple, rápida y evita el problema del gradiente evanescente que tiene la sigmoide en capas ocultas. La sigmoide se reserva solo para la capa de salida (donde necesitamos probabilidades en $[0,1]$).

**Batch Normalization** — normaliza las activaciones de cada capa durante el entrenamiento. Acelera la convergencia y actúa como regularizador suave.

**Dropout** — durante el entrenamiento, aleatoriamente "apaga" un porcentaje de neuronas en cada paso. Previene el *overfitting* forzando a la red a no depender de neuronas específicas.

**Función de pérdida: Binary Cross-Entropy**

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

Es la pérdida estándar para clasificación binaria. Penaliza más severamente las predicciones muy erróneas (ej. predecir $\hat{y} = 0.01$ cuando $y = 1$).

In [ ]:
n_features = X_train.shape[1]

model = keras.Sequential([
    layers.Input(shape=(n_features,)),

    # Bloque 1
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Bloque 2
    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Capa de salida: 1 neurona + sigmoide → P(fraude)
    layers.Dense(1, activation="sigmoid"),
], name="detector_fraude")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ]
)

model.summary()

---
## 7. Entrenamiento

### Parámetros del entrenamiento

- **`epochs=50`**: número de veces que el modelo verá todo el dataset de entrenamiento. Usamos `EarlyStopping` para detener antes si el modelo deja de mejorar.
- **`batch_size=512`**: número de ejemplos procesados en cada actualización de pesos. Batches grandes → más estable pero más lento; batches pequeños → más ruido pero mejor generalización.
- **`class_weight`**: le decimos a Keras que pese más cada ejemplo de fraude.
- **`EarlyStopping`**: detiene el entrenamiento si `val_auc` no mejora en 5 épocas consecutivas, y restaura los mejores pesos.

In [ ]:
import tensorflow as tf

# Lists all GPUs visible to TensorFlow
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))

for gpu in gpus:
    print(f"Device name: {gpu.name}")

In [ ]:
import numpy as np
print(np.__version__)


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Polars → pandas → .values = NumPy C-contiguous garantizado
_X_tr = df_train.select(FEATURE_COLS).to_pandas().values.astype(np.float32, order="C")
_X_te = df_test.select(FEATURE_COLS).to_pandas().values.astype(np.float32, order="C")
_y_tr = df_train["target"].to_pandas().to_numpy().astype(np.float32)
_y_te = df_test["target"].to_pandas().to_numpy().astype(np.float32)

_sc   = StandardScaler()
_X_tr = _sc.fit_transform(_X_tr).astype(np.float32, order="C")
_X_te = _sc.transform(_X_te).astype(np.float32, order="C")

_n_pos = int(_y_tr.sum())
_n_neg = len(_y_tr) - _n_pos
_sw    = np.where(_y_tr == 1.0,
                  float(len(_y_tr) / (2.0 * _n_pos)),
                  float(len(_y_tr) / (2.0 * _n_neg))).astype(np.float32)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=5,
    restore_best_weights=True, verbose=1
)

historia = model.fit(
    _X_tr, _y_tr,
    sample_weight=_sw,
    validation_data=(_X_te, _y_te),
    epochs=50,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nEntrenamiento detenido en época {len(historia.history['loss'])}")


### 7.1 Curvas de aprendizaje

Las curvas de aprendizaje muestran cómo evolucionan la pérdida y el AUC a lo largo del entrenamiento, tanto en train como en validación. Son la herramienta principal para diagnosticar:

- **Underfitting**: ambas curvas convergen a valores malos (el modelo es demasiado simple)
- **Overfitting**: la curva de train sigue mejorando pero la de validación se estanca o empeora
- **Convergencia sana**: ambas curvas mejoran y convergen a valores similares

In [ ]:
hist = historia.history
epocas_plot = list(range(1, len(hist["loss"]) + 1))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Pérdida (Binary Cross-Entropy)", "AUC-ROC")
)

for col_idx, (metric, title) in enumerate([("loss", "Pérdida"), ("auc", "AUC")]):
    col = col_idx + 1
    fig.add_trace(go.Scatter(
        x=epocas_plot, y=hist[metric], name=f"Train — {title}",
        line=dict(color="#3498db", width=2.5)
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=epocas_plot, y=hist[f"val_{metric}"], name=f"Val — {title}",
        line=dict(color="#e74c3c", width=2.5, dash="dash")
    ), row=1, col=col)

# Marcar la mejor época
mejor_epoca = int(np.argmax(hist["val_auc"])) + 1
mejor_auc   = max(hist["val_auc"])
fig.add_vline(
    x=mejor_epoca, line_dash="dot", line_color="gray",
    annotation_text=f"Mejor época: {mejor_epoca}",
    annotation_position="top right"
)

fig.update_layout(
    height=400, width=900,
    title_text="Curvas de aprendizaje",
    title_font_size=16,
    plot_bgcolor="#f8f9fa",
    legend=dict(x=0.01, y=0.01)
)
fig.update_xaxes(title_text="Época")
fig.show()

print(f"Mejor AUC en validación: {mejor_auc:.4f} (época {mejor_epoca})")

---
## 8. Evaluación del modelo

Evaluamos el modelo sobre el **conjunto de test** — datos que el modelo nunca vio durante el entrenamiento.

### El umbral de decisión

La capa de salida produce una **probabilidad** $\hat{p} \in [0,1]$. Para obtener una predicción binaria, aplicamos un umbral $\tau$:

$$\hat{y} = \begin{cases} 1 & \text{si } \hat{p} \geq \tau \\ 0 & \text{si } \hat{p} < \tau \end{cases}$$

El umbral por defecto es $\tau = 0.5$, pero en fraude podemos ajustarlo: bajar $\tau$ aumenta el recall (capturamos más fraudes) a costa de más falsos positivos.

In [ ]:
# Probabilidades predichas
y_prob = model.predict(X_test, verbose=0).flatten()

# Predicción con umbral por defecto 0.5
UMBRAL = 0.5
y_pred = (y_prob >= UMBRAL).astype(int)

print("=" * 55)
print("   REPORTE DE CLASIFICACIÓN (umbral = 0.5)")
print("=" * 55)
print(classification_report(
    y_test, y_pred,
    target_names=["Legítima (0)", "Fraude (1)"],
    digits=4
))
print()
print("Interpretación de las métricas:")
print("  Precision (fraude): de todas nuestras alertas, ¿qué % son fraude real?")
print("  Recall    (fraude): de todos los fraudes reales, ¿qué % detectamos?")
print("  F1-score:           media armónica de precision y recall")

### 8.1 Matriz de confusión

La **matriz de confusión** desglosa las predicciones en cuatro categorías:

|  | Predijo Legítima | Predijo Fraude |
|---|---|---|
| **Es Legítima** | Verdadero Negativo (TN) | Falso Positivo (FP) |
| **Es Fraude** | Falso Negativo (FN) | Verdadero Positivo (TP) |

En detección de fraude, el **FN (fraude no detectado)** es el error más costoso.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"Verdaderos Negativos (TN) — Legítimas bien clasificadas: {tn:>10,}")
print(f"Falsos Positivos    (FP) — Legítimas marcadas como fraude: {fp:>8,}")
print(f"Falsos Negativos    (FN) — Fraudes NO detectados:           {fn:>8,}  ← crítico")
print(f"Verdaderos Positivos(TP) — Fraudes detectados:              {tp:>8,}")

etiquetas = [
    [f"TN\n{tn:,}", f"FP\n{fp:,}"],
    [f"FN ⚠️\n{fn:,}", f"TP\n{tp:,}"]
]

fig = go.Figure(go.Heatmap(
    z=cm,
    x=["Predijo Legítima", "Predijo Fraude"],
    y=["Es Legítima", "Es Fraude"],
    text=etiquetas,
    texttemplate="%{text}",
    textfont=dict(size=16),
    colorscale=[[0, "#eaf4fb"], [1, "#1a5276"]],
    showscale=False
))
fig.update_layout(
    title=dict(text="Matriz de confusión", font_size=16),
    xaxis_title="Predicción",
    yaxis_title="Valor real",
    width=450, height=380
)
fig.show()

### 8.2 Curva ROC y AUC

La **curva ROC** (Receiver Operating Characteristic) muestra la relación entre:
- **TPR (Recall)**: fracción de fraudes reales detectados
- **FPR (Tasa de Falsos Positivos)**: fracción de legítimas incorrectamente marcadas

para todos los umbrales posibles $\tau \in [0, 1]$. El **AUC** (Área Bajo la Curva) resume la curva en un solo número:
- AUC = 1.0 → clasificador perfecto
- AUC = 0.5 → clasificador aleatorio (diagonal)
- AUC < 0.5 → peor que aleatorio

In [ ]:
fpr, tpr, umbrales_roc = roc_curve(y_test, y_prob)
auc_roc = auc(fpr, tpr)

fig = go.Figure()

# Clasificador aleatorio (referencia)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(color="gray", dash="dot", width=1.5),
    name="Aleatorio (AUC = 0.50)"
))

# Curva ROC del modelo
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    line=dict(color="#2980b9", width=3),
    fill="tonexty", fillcolor="rgba(41,128,185,0.12)",
    name=f"Modelo (AUC = {auc_roc:.4f})"
))

# Marcar el punto correspondiente al umbral 0.5
idx_umbral = np.argmin(np.abs(umbrales_roc - UMBRAL))
fig.add_trace(go.Scatter(
    x=[fpr[idx_umbral]], y=[tpr[idx_umbral]], mode="markers",
    marker=dict(size=14, color="#e74c3c", symbol="x"),
    name=f"Umbral = {UMBRAL}"
))

fig.update_layout(
    title=dict(text=f"Curva ROC — AUC = {auc_roc:.4f}", font_size=16),
    xaxis=dict(title="Tasa de Falsos Positivos (FPR)", range=[0, 1]),
    yaxis=dict(title="Tasa de Verdaderos Positivos (TPR / Recall)", range=[0, 1.02]),
    plot_bgcolor="#f8f9fa",
    width=580, height=500,
    legend=dict(x=0.42, y=0.08)
)
fig.show()
print(f"AUC-ROC: {auc_roc:.4f}")

### 8.3 Curva Precision-Recall

Para datasets muy desbalanceados, la curva **Precision-Recall** es más informativa que la ROC. Mientras la ROC evalúa a todos los umbrales qué tan bien distingue el modelo, la curva PR se enfoca en la **clase minoritaria** (fraude).

Un clasificador que siempre predice "fraude" tendría Recall=1 pero Precision muy baja. Un clasificador perfecto tiene Precision=1 y Recall=1 en todo momento.

El **AUC-PR** (o *Average Precision*) es la métrica de referencia en papers de detección de fraude.

In [ ]:
precision_vals, recall_vals, umbrales_pr = precision_recall_curve(y_test, y_prob)
auc_pr = average_precision_score(y_test, y_prob)

# Línea base: un clasificador aleatorio tiene Precision ≈ tasa de fraude
tasa_fraude_test = y_test.mean()

fig = go.Figure()

# Referencia: clasificador aleatorio
fig.add_hline(
    y=tasa_fraude_test,
    line_dash="dot", line_color="gray",
    annotation_text=f"Aleatorio (AP ≈ {tasa_fraude_test:.3f})",
    annotation_position="top right"
)

# Curva PR del modelo
fig.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals, mode="lines",
    line=dict(color="#8e44ad", width=3),
    fill="tonexty", fillcolor="rgba(142,68,173,0.10)",
    name=f"Modelo (AP = {auc_pr:.4f})"
))

# Marcar punto con umbral 0.5
idx_pr = np.argmin(np.abs(umbrales_pr - UMBRAL))
fig.add_trace(go.Scatter(
    x=[recall_vals[idx_pr]], y=[precision_vals[idx_pr]], mode="markers",
    marker=dict(size=14, color="#e74c3c", symbol="x"),
    name=f"Umbral = {UMBRAL}"
))

fig.update_layout(
    title=dict(text=f"Curva Precision-Recall — Average Precision = {auc_pr:.4f}", font_size=16),
    xaxis=dict(title="Recall (fracción de fraudes detectados)", range=[0, 1]),
    yaxis=dict(title="Precision (fracción de alertas correctas)", range=[0, 1.05]),
    plot_bgcolor="#f8f9fa",
    width=580, height=500,
    legend=dict(x=0.02, y=0.15)
)
fig.show()
print(f"Average Precision (AUC-PR): {auc_pr:.4f}")
print(f"Ratio de mejora sobre baseline aleatorio: {auc_pr/tasa_fraude_test:.1f}x")

### 8.4 Selección del umbral óptimo

El umbral por defecto (0.5) no es necesariamente el mejor. Dependiendo del costo de cada tipo de error, podemos elegir un umbral diferente. Una heurística común es el **umbral F1-óptimo**: aquel que maximiza el F1-score sobre la clase de fraude.

In [ ]:
# Calcular F1 para cada umbral en la curva PR
# Evitar división por cero
denom = precision_vals[:-1] + recall_vals[:-1]
denom = np.where(denom == 0, 1e-9, denom)
f1_por_umbral = 2 * precision_vals[:-1] * recall_vals[:-1] / denom

idx_mejor_f1 = np.argmax(f1_por_umbral)
umbral_optimo = float(umbrales_pr[idx_mejor_f1])
mejor_f1 = f1_por_umbral[idx_mejor_f1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=umbrales_pr, y=f1_por_umbral, mode="lines",
    line=dict(color="#27ae60", width=2.5), name="F1 por umbral"
))
fig.add_vline(
    x=umbral_optimo, line_dash="dash", line_color="#e74c3c",
    annotation_text=f"Umbral óptimo = {umbral_optimo:.3f}\nF1 = {mejor_f1:.4f}",
    annotation_position="top left"
)
fig.update_layout(
    title="F1-score por umbral de decisión",
    xaxis_title="Umbral τ",
    yaxis_title="F1-score (clase fraude)",
    plot_bgcolor="#f8f9fa",
    width=650, height=380
)
fig.show()

# Reporte con el umbral óptimo
y_pred_optimo = (y_prob >= umbral_optimo).astype(int)
print(f"Reporte con umbral óptimo ({umbral_optimo:.3f}):")
print(classification_report(
    y_test, y_pred_optimo,
    target_names=["Legítima (0)", "Fraude (1)"],
    digits=4
))

---
## 9. Conclusiones

### Resumen del pipeline

```
CSV crudo
  → Polars (scan_csv + LazyFrame)
  → Limpieza + ingeniería de features (Amount, Time, cíclicas, one-hot, MCC)
  → Split estratificado train/test  ← antes del target encoding
  → Target encoding (fit solo en train)
  → StandardScaler (fit solo en train)
  → Keras: Dense(64) → Dense(32) → Dense(1) + class_weight
  → Evaluación: AUC-ROC, AUC-PR, Precision, Recall, umbral óptimo
```

### Lecciones clave

1. **El orden importa.** El split debe hacerse antes de cualquier transformación que use el target (target encoding, estadísticas por grupo). Hacerlo después introduce data leakage.

2. **Accuracy es inútil en fraude.** Un modelo que nunca detecta fraude puede tener >99% de accuracy. Usamos AUC-ROC, AUC-PR, Recall y Precision.

3. **`class_weight` es fundamental.** Sin él, la red converge trivialmente a "siempre predice legítima". Darle más peso a la clase minoritaria es la intervención más simple y efectiva.

4. **El umbral no es fijo.** Ajustar $\tau$ permite mover el balance Precision/Recall según las necesidades del negocio.

5. **Polars + Keras = pipeline moderno y eficiente.** Polars maneja la transformación tabular a gran escala; Keras solo recibe arrays NumPy. La separación de responsabilidades es clara.

---

### Ejercicios para el estudiante

1. **Feature engineering adicional:**  
   - Añadir `minute_sin` / `minute_cos` — ¿mejora el modelo?
   - Crear el ratio `amount / (promedio_monto_por_mcc)` — ¿qué captura esta feature?

2. **Arquitectura:**  
   - ¿Qué pasa si eliminas el `BatchNormalization`? ¿Y el `Dropout`?
   - Prueba una arquitectura más profunda: 128 → 64 → 32 → 1. ¿Mejora?

3. **Desbalance:**  
   - Instala `imbalanced-learn` y aplica SMOTE para sobremuestrear la clase de fraude. ¿Cómo cambian las métricas?

4. **Umbral:**  
   - Supón que detectar un fraude vale \$100 y bloquear una transacción legítima cuesta \$5. ¿Cuál sería el umbral óptimo desde el punto de vista económico?

5. **Línea base:**  
   - Entrena un `RandomForestClassifier` de scikit-learn con las mismas features. ¿Supera a la red neuronal en AUC-PR? ¿Por qué sería o no sería sorprendente?